## Project 1

### Eliminating Child Care Deserts in New York State through Optimization


In [2]:
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import folium

In [ ]:
# 1) Read data files
avg_individual_income = pd.read_csv('ChildCareDeserts_Data/avg_individual_income.csv')
avg_individual_income.head()

child_care_regulated = pd.read_csv('ChildCareDeserts_Data/child_care_regulated.csv')
child_care_regulated.head()

employment_rate = pd.read_csv('ChildCareDeserts_Data/employment_rate.csv')
employment_rate.head()

population = pd.read_csv('ChildCareDeserts_Data/population.csv')
population.head()

potential_locations = pd.read_csv('ChildCareDeserts_Data/potential_locations.csv')
potential_locations.head()

,zipcode,latitude,longitude
0,501,40.816376,-73.040796
1,501,40.817455,-73.044202
2,501,40.813873,-73.042182
3,501,40.814152,-73.037265
4,501,40.824673,-73.047431


In [4]:
# 2) Standardize ZIP column names to `zipcode`:
#    - income: rename 'ZIP code' -> 'zipcode'
#    - childcare: rename 'zip_code' -> 'zipcode'
#    - ensure others are already 'zipcode' or rename accordingly

# Print current column names for each dataframe
print("Income columns:", avg_individual_income.columns.tolist())
print("Childcare columns:", child_care_regulated.columns.tolist())
print("Employment columns:", employment_rate.columns.tolist())
print("Population columns:", population.columns.tolist())
print("Potential locations columns:", potential_locations.columns.tolist())

# Standardize ZIP code columns
avg_individual_income = avg_individual_income.rename(columns={'ZIP code': 'zipcode'})
child_care_regulated = child_care_regulated.rename(columns={'zip_code': 'zipcode'})

# Verify the changes
print("\nAfter standardization:")
print("Income columns:", avg_individual_income.columns.tolist())
print("Childcare columns:", child_care_regulated.columns.tolist())
print("Employment columns:", employment_rate.columns.tolist())
print("Population columns:", population.columns.tolist())
print("Potential locations columns:", potential_locations.columns.tolist())

Income columns: ['ZIP code', 'average income']
Childcare columns: ['facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zip_code', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude']
Employment columns: ['zipcode', 'employment rate']
Population columns: ['zipcode', 'Total', '-5', '5-9', '10-14', '15-19', '20-24', '25-29', '30-34', '35-39', '40-44', '45-49', '50-54', '55-59', '60-64', '65-69', '70-74', '75-79', '80-84', '85+']
Potential locations columns: ['zipcode', 'latitude', 'longitude']

After standardization:
Income columns: ['zipcode', 'average income']
Childcare columns: ['facility_id', 'program_type', 'facility_status', 'facility_name', 'city', 'zipcode', 'school_district_name', 'infant_capacity', 'toddler_capacity', 'preschool_capacity', 'school_age_capacity', 'children_capacity', 'total_capacity', 'latitude', 'longitude']
Employment co

In [ ]:
# 3) Force ZIPs to be strings, zero-pad to 5 digits:
dataframes = [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]
for df in dataframes:
    df['zipcode'] = df['zipcode'].astype(str).str.zfill(5)

# Verify ZIP code formatting
for i, df in enumerate(dataframes, start=1):
    print(f"\nDataFrame {i} ZIP codes sample:", df['zipcode'].head())




DataFrame 1 ZIP codes sample: 0    10001
1    10002
2    10003
3    10004
4    10005
Name: zipcode, dtype: object

DataFrame 2 ZIP codes sample: 0    13323
1    14701
2    12590
3    10960
4    12077
Name: zipcode, dtype: object

DataFrame 3 ZIP codes sample: 0    10001
1    10002
2    10003
3    10004
4    10005
Name: zipcode, dtype: object

DataFrame 4 ZIP codes sample: 0    06390
1    10001
2    10002
3    10003
4    10004
Name: zipcode, dtype: object

DataFrame 5 ZIP codes sample: 0    00501
1    00501
2    00501
3    00501
4    00501
Name: zipcode, dtype: object


In [ ]:
# 4) Basic sanity checks:
#    - assert no null ZIPs after padding
#    - preview frames with head() to confirm expected columns

# Check for null values in zipcode columns
print("Checking for null ZIP codes:")
for name, df in zip(['Income', 'Childcare', 'Employment', 'Population', 'Potential Locations'], 
                   [avg_individual_income, child_care_regulated, employment_rate, population, potential_locations]):
    null_count = df['zipcode'].isnull().sum()
    print(f"{name} dataset: {null_count} null ZIP codes")
    # Assert no null values
    assert null_count == 0, f"Found {null_count} null ZIP codes in {name} dataset"


Checking for null ZIP codes:
Income dataset: 0 null ZIP codes
Childcare dataset: 0 null ZIP codes
Employment dataset: 0 null ZIP codes
Population dataset: 0 null ZIP codes
Potential Locations dataset: 0 null ZIP codes


In [15]:
# Analyze childcare facility sizes
# Define size categories based on total capacity
def get_size_category(capacity):
    if capacity <= 100:
        return 'Small'
    elif capacity <= 200:
        return 'Medium'
    elif capacity <= 400:
        return 'Large'
    else:
        return 'XL'

# Add size category column
child_care_regulated['size_category'] = child_care_regulated['total_capacity'].apply(get_size_category)

# Get counts for each category
size_counts = child_care_regulated['size_category'].value_counts().sort_index()
print("Number of facilities in each size category:")
print(size_counts)

# Calculate percentage distribution
size_percentages = (size_counts / len(child_care_regulated) * 100).round(1)
print("\nPercentage distribution:")
for category, percentage in size_percentages.items():
    print(f"{category}: {percentage}%")

# Show summary statistics for each category
print("\nCapacity statistics by category:")
print(child_care_regulated.groupby('size_category')['total_capacity'].agg(['min', 'max', 'mean', 'count']))

Number of facilities in each size category:
size_category
Large       493
Medium     1414
Small     13647
XL           50
Name: count, dtype: int64

Percentage distribution:
Large: 3.2%
Medium: 9.1%
Small: 87.5%
XL: 0.3%

Capacity statistics by category:
               min  max        mean  count
size_category                             
Large          201  400  261.667343    493
Medium         101  200  141.959689   1414
Small            0  100   22.424562  13647
XL             405  890  499.080000     50
